# Задание 1 (2 балла)


Выберите 5 языков в википедии (не тех, что использовались в семинаре). Скачайте по 10 случайных статей для каждого языка. Предобработайте тексты, удаляя лишние теги/отступы/разделители (если они есть). Разделите тексты на предложения и создайте датасет, в котором каждому предложению соответствует язык. Кластеризуйте тексты, используя эбмединг модель из прошлого семинара и любой алгоритм кластеризации. Проверьте качество кластеризации с помощь метрики ARI. Отдельно проанализируйте 3 ошибочно кластеризованных текста (если такие есть).

In [41]:
%pip install wikipedia fasttext

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.1.0-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.1.0-py3-none-any.whl (319 kB)
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11756 sha256=26762ddb24d7250d80420a8066172bf6e2b147ebf1f650234b51796a0a7e504b
  Stored in directory: /root/.cache/pip/wheels/79/1d/c8/b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
  Created wheel for fasttext: filename=fasttext-0.9.3-cp313-cp313-linux_x86_64.whl size=4711564 sha256=ccc0be61863d42b8fe916ca09512e2da4bde3336959e81a5bfc19eaa2f9f80f7
  Stored in directory: /root/.cache/pip/wheels/59/06/3f/c95dbba0df6e58ba6ed18220c1aea1e96042802139df3674ef
Successfully built wikipedia fasttext


In [42]:
import json
import math
import os
import re
import time
from collections import defaultdict, Counter
from itertools import chain
import nltk
import numpy as np
import pandas as pd
import regex
import requests
import wikipedia
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from tqdm import tqdm

nltk.download('punkt_tab')
wikipedia.set_user_agent('HSE-NLP-course/0.1 (sherstneva.katja@gmail.com)')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


### Загрузка текстов

In [43]:
MIN_INTERVAL = 0.35 # секунд между запросами -> не больше ~170 запросов в минуту
_last_call = [0.0]

def _wiki_request(params):
    params['format'] = 'json'
    params.setdefault('action', 'query')
    for attempt in range(5):
        wait = MIN_INTERVAL - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.time()
        r = requests.get(wikipedia.wikipedia.API_URL, params=params,
                         headers={'User-Agent': wikipedia.wikipedia.USER_AGENT}, timeout=30)
        if r.status_code == 429: # Too Many Requests - ждем, сколько просит сервер
            retry_after = int(r.headers.get('Retry-After', 30))
            print(f'429 Too Many Requests, ждем {retry_after} с')
            time.sleep(retry_after)
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

# все функции библиотеки (random, page, languages...) теперь ходят через нашу версию
wikipedia.wikipedia._wiki_request = _wiki_request

Языки: саамские и прибалтийско-финские. Все на латинице, часто содержат похожие символы (например, умляуты).

* `'smn'` -- инари-саамский
* `'se'` -- северносаамский
* `'fi'` -- финский
* `'et'` -- эстонский
* `'vep'` -- веппский



In [44]:
langs = ['smn', 'se', 'fi', 'et', 'vep'] # Inari Saami, North Saami, Finnish, Estonian, Vepps

def load_with_disambigution(page):
    try:
        p = wikipedia.page(page)
    except wikipedia.DisambiguationError as e:
        p = wikipedia.page(np.random.choice(e.options))
    return p


def get_texts_for_lang(lang, n=100, min_len=200):
    wikipedia.set_lang(lang)
    wiki_content = []

    while len(wiki_content) < n:
        try:
            pages = wikipedia.random(10)
        except Exception as e:
            print(e)
            time.sleep(2)
            continue

        for page_name in pages:
            if len(wiki_content) >= n:
                break
            for attempt in range(6): # повторные попытки
                try:
                    page = load_with_disambigution(page_name)
                    content = page.content.replace('==', '')
                    if len(content) >= min_len:
                        wiki_content.append(f'{page.title}\n{content}')
                        time.sleep(1)
                    break
                except (wikipedia.DisambiguationError, wikipedia.PageError) as e:
                    print(f'Skipping page {page_name}: {e}')
                    break
                except Exception as e:
                    if attempt == 5:
                        print(f'Skipping page {page_name}: {e}')
                    else:
                        time.sleep(2 ** (attempt + 1))
    return wiki_content

# кэш
wiki_texts = (json.load(open('wiki_texts.json', encoding='utf-8')) if os.path.exists('wiki_texts.json') else {})

for lang in langs:
    if wiki_texts.get(lang):
        continue
    try:
        wiki_texts[lang] = get_texts_for_lang(lang, n=10)
        with open('wiki_texts.json', 'w', encoding='utf-8') as f:
            json.dump(wiki_texts, f, ensure_ascii=False)
        print(lang, len(wiki_texts[lang]))
    except Exception as e:
        print('ERROR ON', lang, str(e).strip())

smn 10
se 10


/usr/local/lib/python3.13/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.13/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


fi 10
et 10
vep 10


In [45]:
def cleaning(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return sent_tokenize(text)

rows = []
for lang, articles in wiki_texts.items():
    for article in articles:
        _, _, body = article.partition('\n')
        for sent in cleaning(body):
            if len(sent) > 3:
                rows.append({'sentence': sent, 'lang': lang})

df = pd.DataFrame(rows).drop_duplicates(subset='sentence').reset_index(drop=True)

df.head()

,sentence,lang
0,"Kemiallâš merkkâ lii uánihis, läättinlâš pusta...",smn
1,Puoh luándu algâamnâsáid láá oovtâ teikkâ kyev...,smn
2,Kemiallâš ovtâstuvâi kemiallâš myenster lii ra...,smn
3,Ko algâamnâs kemiallâš meerhâst láá kyehti puu...,smn
4,"Ovdâmerkkân Co lii kooboolt kemiallâš merkkâ, ...",smn


In [48]:
len(df)

964

### Кластеризация

In [46]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import adjusted_rand_score

import fasttext

In [47]:
data = df[df['sentence'].str.len() >= 20].reset_index(drop=True)
sentences = data['sentence'].tolist()
labels = data['lang'].values

In [50]:
!wget -O lid.176.ftz https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz
lid_model = fasttext.load_model('lid.176.ftz')

--2026-10-13 10:41:11--  https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.230.79.93, 54.230.79.53, 54.230.79.88, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.230.79.93|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 938013 (916K) [binary/octet-stream]
Saving to: ‘lid.176.ftz’

lid.176.ftz         100%[===================>] 916.03K  --.-KB/s    in 0.06s   

2026-10-13 10:41:12 (14.5 MB/s) - ‘lid.176.ftz’ saved [938013/938013]



In [51]:
X = np.array([lid_model.get_sentence_vector(s.replace('\n', ' ')) for s in sentences])
X = normalize(X)

In [52]:
k = len(set(labels))
km = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = km.fit_predict(X)

#### Оценка

In [53]:
# оценка
ari = adjusted_rand_score(labels, clusters)
print(f'ARI = {ari:.3f}')

ARI = 0.535


#### Анализ ошибок

In [58]:
# язык кластера = самый частый язык текстов
cluster_to_lang = {}
for cluster_id in set(clusters):
    cluster_labels = labels[clusters == cluster_id]
    cluster_to_lang[cluster_id] = Counter(cluster_labels).most_common(1)[0][0]

errors = [i for i in range(len(sentences)) if labels[i] != cluster_to_lang[clusters[i]]]
len(errors)

250

In [59]:
results = pd.DataFrame({'true_lang': labels, 'cluster_lang': [cluster_to_lang[c] for c in clusters]})
results['error'] = results['true_lang'] != results['cluster_lang']

stats = (results.groupby('true_lang').agg(
        total=('error', 'size'),
        errors=('error', 'sum')))

stats['error_%'] = (100 * stats['errors'] / stats['total']).round(2)
stats = stats.sort_values('error_%', ascending=False)


In [60]:
stats

,total,errors,error_%
true_lang,,,
smn,119,119,100.00
vep,221,69,31.22
se,131,39,29.77
et,213,15,7.04
fi,257,8,3.11


Инари-саамскиим предложениям вообще не формируются похожие эмбеддинги -- всё рассовывается по "чужим" кластерам.

In [65]:
fi_errors = results[(results['true_lang'] == 'fi') & results['error']]

for i in fi_errors.index:
    print(f"\nЯзык кластера: {fi_errors.loc[i, 'cluster_lang']}")
    print(f"Текст: {sentences[i]}")


Язык кластера: et
Текст: B. Baillaud, H. Bourget (toim.

Язык кластера: et
Текст: G. Michelacci (toim.

Язык кластера: vep
Текст: ): ”Le Lettere di Charles Hermite a Angelo Genocchi (1868–1887)”, Guaderni matematci II, s. 546.

Язык кластера: et
Текст: (ranskaksi) Catherine Goldstein, Norbert Schappacher, Joachim Scmwermer: ”The Hermitian Form of Reading the Disquisitiones”, The Shaping of Arithmetic, s. 377–410.

Язык кластера: et
Текст: D. Flament, P. Nabonnand (toim.

Язык кластера: vep
Текст: (ranskaksi) Catherie Goldstein: Charles Hermite's Stroll through the Galois fields.

Язык кластера: vep
Текст: Leningrad: Nauka, 1982.

Язык кластера: et
Текст: Alarik Nysténin vanhemmat olivat Joensuun pormestari Johan Alfred Nystén ja Fanny Vilhelmina Holmström.


Логично. В этих предложениях мало собственно финского текста -- в основном фамилии / названия произведений (иногда на английском).

Только последнее более-менее содержит финский текст, но его тоже очень мало.

## Задание 2 (2 балла)
Возьмите какой-нибудь из файлов из Syntagrus или Taiga -
https://github.com/UniversalDependencies/UD_Russian-SynTagRus/tree/master   
https://tatianashavrina.github.io/taiga_site/downloads  

Используя Dependency Matcher напишите 2 разных query к этому корпусу, результаты которых выдают что-то осмысленное (показывают какую-то закономерность; подвсечивают примеры какого-то феномена; выдают то, что можно использовать в другой задаче).
Добавьте дополнительные правила фильтрации, если получается слишком зашумленный или наоборот пустой результат (см. семинар и примеры для SVO)
Используйте именно синтактическую структуру, а не просто слова и POS тэги.
Обязательно покажите примеры, которые находятся по вашим запросам.


Ответ должен быть структурирован как-то так:
```
## query
Краткое описание того, что вы пытаетесь найти

### код
код с запросом (или несколько ячеек где вы последовательно улучшаете запрос)

### примеры
несколько самых показательных примеров или даже какая-то агрегация вроде топ-N

... (и так два раза)
```



In [66]:
!pip install conllu
import conllu
import os
import conllu
from collections import Counter
from spacy.matcher import DependencyMatcher
import spacy
from spacy.tokens import Doc

!wget https://linghub.ru/static/Taiga/Fake_news.rar
!unrar x -o+ Fake_news.rar

--2026-10-13 11:00:13--  https://linghub.ru/static/Taiga/Fake_news.rar
Resolving linghub.ru (linghub.ru)... 89.175.46.138
Connecting to linghub.ru (linghub.ru)|89.175.46.138|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3712692 (3.5M) [application/rar]
Saving to: ‘Fake_news.rar.1’

Fake_news.rar.1     100%[===================>]   3.54M   782KB/s    in 5.7s    

2026-10-13 11:00:20 (637 KB/s) - ‘Fake_news.rar.1’ saved [3712692/3712692]


UNRAR 7.00 freeware      Copyright (c) 1993-2024 Alexander Roshal


Extracting from Fake_news.rar

Extracting  Panorama/metatable.csv                                         0%  1%  OK 
Extracting  Panorama/tagged_texts/0.txt                                    1%  OK 
Extracting  Panorama/tagged_texts/1.txt                                    1%  OK 
Extracting  Panorama/tagged_texts/10.txt                                   2%  OK 
Extracting  Panorama/tagged_texts/100.txt               

In [4]:
def conllu_to_doc(sentence):
    words = [t for t in sentence if isinstance(t['id'], int)]
    if not words:
        return None
    position = {t['id']: k for k, t in enumerate(words)} # id в CoNLL-U -> позиция в Doc
    return Doc(
        nlp_blank.vocab,
        words=[t['form'] for t in words],
        spaces=[(t['misc'] or {}).get('SpaceAfter') != 'No' for t in words],
        # у корня в spaCy вершина - он сам
        heads=[position.get(t['head'], k) for k, t in enumerate(words)],
        deps=['ROOT' if t['head'] == 0 else t['deprel'] for t in words],
        pos=[t['upos'] for t in words],
        lemmas=[t['lemma'] for t in words],
        morphs=['|'.join(f'{k}={v}' for k, v in (t['feats'] or {}).items()) for t in words],
    )

In [34]:
nlp_blank = spacy.blank('ru')

path = 'Panorama/tagged_texts/'
files = sorted(os.listdir(path))

# предпросмотр первого файла
with open(os.path.join(path, files[1]), encoding='utf-8') as f:
    print(f.read()[:1200])

docs = {}  # имя файла -> список Doc (по одному на предложение)

for file in files:
    with open(os.path.join(path, file), encoding='utf-8') as f:
        sentences = conllu.parse(f.read())
    docs[file] = [d for d in map(conllu_to_doc, sentences) if d is not None]

all_docs = [d for docs in docs.values() for d in docs]
print('файлов:', len(docs), '| предложений:', len(all_docs))


# sent_id = 11
# text = Китайская провинция Цзянси, по решению ЦК КПК, станет пилотным регионом страны, в котором буде целиком ликвидировано хождение американского доллара.
1	Китайская	китайский	ADJ	_	Case=Nom|Degree=Pos|Gender=Fem|Number=Sing	2	amod	_	_
2	провинция	провинция	NOUN	_	Animacy=Inan|Case=Nom|Gender=Fem|Number=Sing	10	nsubj	_	_
3	Цзянси	Цзяньси	PROPN	_	Animacy=Inan|Case=Nom|Gender=Fem|Number=Sing	5	appos	_	SpaceAfter=No
4	,	,	PUNCT	_	_	3	punct	_	_
5	по	по	ADP	_	_	6	case	_	_
6	решению	решение	NOUN	_	Animacy=Inan|Case=Dat|Gender=Neut|Number=Sing	2	nmod	_	_
7	ЦК	цк	PROPN	_	Animacy=Inan|Case=Gen|Gender=Masc|Number=Sing	6	nmod	_	_
8	КПК	кпк	PROPN	_	Animacy=Inan|Case=Gen|Gender=Fem|Number=Sing	7	nmod	_	SpaceAfter=No
9	,	,	PUNCT	_	_	8	punct	_	_
10	станет	становиться	VERB	_	Aspect=Perf|Mood=Ind|Number=Sing|Person=3|Tense=Fut|VerbForm=Fin|Voice=Mid	0	root	_	_
11	пилотным	пилотный	ADJ	_	Case=Ins|Degree=Pos|Gender=Masc|Number=Sing	12	amod	_	_
12	регионом	регион	NOUN	_	Animacy=Inan|Cas

## Query 1

In [96]:
def test_matcher(matcher, docs, max_dist=3, n=5):
    results = []
    for doc in docs:
        for match_id, ids in matcher(doc):
            name = doc.vocab.strings[match_id]
            noun, num = ids
            if abs(num - noun) > max_dist:
                continue
            order = 'num n' if num < noun else 'n num'
            results.append((doc, num, noun, order, name))

    print(f'Всего {len(results)}')

    for label in ['num n', 'n num']:
        print(f'\n\n{label}')
        shown = [r for r in results if r[3] == label][:n]
        for doc, num, noun, order, name in shown:
            start = min(num, noun)
            end = max(num, noun) + 1
            print(
                f'{doc[start:end].text} [{name}] | '
                f'NUM: {doc[num].text}/{doc[num].dep_} | '
                f'NOUN: {doc[noun].text}/{doc[noun].dep_}'
            )

    return results

В этом запросе я ищу группы с количественными числительными, чтобы извлечь группы с обычным (пять книг) и инвертированным (книг пять) порядком.

Группы с разными числительными устроены по-разному:
* после 5 -- управление, числительное -- вершна, управляет генитивом (пять столов)
* до 5 -- согласование, сущ. -- вершина, числительное и сущ. согласуются по падежу (два стола)

In [86]:
matcher_num = DependencyMatcher(all_docs[0].vocab)

# существительное -- вершина, числительное -- зависимое (nummod)
matcher_num.add('NOUN_HEAD', [[
    {'RIGHT_ID': 'noun', 'RIGHT_ATTRS': {'POS': 'NOUN'}},
    {'LEFT_ID': 'noun', 'REL_OP': '>', 'RIGHT_ID': 'num',
     'RIGHT_ATTRS': {'POS': 'NUM', 'DEP': 'nummod'}}]])

# числительное -- вершина, существительное -- зависимое (nummod:gov)
matcher_num.add('NUM_HEAD', [[
    {'RIGHT_ID': 'num', 'RIGHT_ATTRS': {'POS': 'NUM'}},
    {'LEFT_ID': 'num', 'REL_OP': '>', 'RIGHT_ID': 'noun',
     'RIGHT_ATTRS': {'POS': {'IN': ['NOUN', 'PROPN']}, 'DEP': 'nummod:gov'}}]])

In [97]:
res1 = test_matcher(matcher_num, all_docs)

Всего 1118


num n
1945 года [NOUN_HEAD] | NUM: 1945/nummod | NOUN: года/nmod
100 миллиардов [NOUN_HEAD] | NUM: 100/nummod | NOUN: миллиардов/obl
трем месячным зарплатам [NOUN_HEAD] | NUM: трем/nummod | NOUN: зарплатам/conj
5 лет [NOUN_HEAD] | NUM: 5/nummod | NOUN: лет/nmod
2017 года [NOUN_HEAD] | NUM: 2017/nummod | NOUN: года/nmod


n num
весной 2020 [NOUN_HEAD] | NUM: 2020/nummod | NOUN: весной/obl
отмены одного [NOUN_HEAD] | NUM: одного/nummod | NOUN: отмены/nmod
движению по одной [NOUN_HEAD] | NUM: одной/nummod | NOUN: движению/obl
жены лучше одной [NOUN_HEAD] | NUM: одной/nummod | NOUN: жены/obl
факультета одного [NOUN_HEAD] | NUM: одного/nummod | NOUN: факультета/nmod


Добавлю согласование в падеже и управление (именно ГЕНИТИВОМ), исключу числительные, записанные цифрами и с леммой "один"

In [101]:
CASES = ['Nom', 'Gen', 'Dat', 'Acc', 'Ins', 'Loc']

matcher_num2 = DependencyMatcher(all_docs[0].vocab)

# согласование
for case in CASES:
    matcher_num2.add(f'AGREE_{case}',
     [[{'RIGHT_ID': 'noun', 'RIGHT_ATTRS': {'POS': 'NOUN', 'MORPH': {'IS_SUPERSET': [f'Case={case}']}}},
        {'LEFT_ID': 'noun', 'REL_OP': '>', 'RIGHT_ID': 'num',
          'RIGHT_ATTRS': {'POS': 'NUM', 'DEP': 'nummod', 'TEXT': {'REGEX': '^[а-яА-ЯёЁ]+$'},
                          'LEMMA': {'NOT_IN': ['один']}, 'MORPH': {'IS_SUPERSET': [f'Case={case}']}}}]])
# управление генитивом
matcher_num2.add('NUM_GOV',
 [[{'RIGHT_ID': 'noun', 'RIGHT_ATTRS': {'POS': 'NOUN', 'MORPH': {'IS_SUPERSET': ['Case=Gen']}}},
    {'LEFT_ID': 'noun', 'REL_OP': '>', 'RIGHT_ID': 'num',
     'RIGHT_ATTRS': {'POS': 'NUM', 'DEP': 'nummod:gov', 'TEXT': {'REGEX': '^[а-яА-ЯёЁ]+$'},
            'LEMMA': {'NOT_IN': ['один']}}}]])

In [102]:
res2 = test_matcher(matcher_num2, all_docs)

Всего 350


num n
трем месячным зарплатам [AGREE_Dat] | NUM: трем/nummod | NOUN: зарплатам/conj
утробе матери [NUM_GOV] | NUM: утробе/nummod:gov | NOUN: матери/obl
несколько тысяч [NUM_GOV] | NUM: несколько/nummod:gov | NOUN: тысяч/nsubj
трёх месяцев [AGREE_Gen] | NUM: трёх/nummod | NOUN: месяцев/nmod
двух десятков [AGREE_Gen] | NUM: двух/nummod | NOUN: десятков/nmod


n num
текстов Тимати [AGREE_Gen] | NUM: Тимати/nummod | NOUN: текстов/nmod
лет пять [NUM_GOV] | NUM: пять/nummod:gov | NOUN: лет/obl


## Query 2

В этом запросе меня интересуют предлоги, которые управляют предложным (Loc) или винительным (Acc) падежом.

In [117]:
matcher_pp1 = DependencyMatcher(all_docs[0].vocab)

for case in ['Acc', 'Loc']:
    matcher_pp1.add(f'PP_{case.upper()}', [[{"RIGHT_ID": "noun", "RIGHT_ATTRS": {
                "POS": {"IN": ["NOUN"]},
                "MORPH": {"IS_SUPERSET": [f"Case={case}"]}}},
        {"LEFT_ID": "noun", "REL_OP": ">", "RIGHT_ID": "prep", "RIGHT_ATTRS": {"DEP": "case"}}]])

In [120]:
matches = []

for doc in all_docs:
    for match_id, ids in matcher_pp1(doc):
        name = doc.vocab.strings[match_id]
        noun, prep = ids
        case_label = name.replace('PP_', '').title()
        matches.append((doc[prep].text.lower(), case_label, doc.text))

counts = Counter((prep, case) for prep, case, _ in matches)
examples = defaultdict(list)

for prep, case, sentence in matches:
    if sentence not in examples[(prep, case)]:
        examples[(prep, case)].append(sentence)

both_cases = sorted({
    prep for prep, case, _ in matches
    if counts[(prep, 'Acc')] and counts[(prep, 'Loc')]
})

for prep in both_cases:
    print(f"\n{prep}: Acc = {counts[(prep, 'Acc')]}, Loc = {counts[(prep, 'Loc')]}")
    for case in ['Acc', 'Loc']:
        print(f"{case}:")
        for sentence in examples[(prep, case)][:1]:
            print(f"{sentence}")


без: Acc = 4, Loc = 2
Acc:
Ни один солдат не покинет бой без особого приказа.
Loc:
Табак для "Патриарха" планируют выращивать исключительно на монастырских угодьях без использования какой-либо техники, а каждая сигарета будет создана иноками вручную.

в: Acc = 903, Loc = 2311
Acc:
В город прорывается группа диверсантов под командованием Анри Жореса, накануне совершившего каминг-аут.
Loc:
Действие ленты происходит в апреле 1945 года.

во: Acc = 15, Loc = 35
Acc:
Конечно мы будем бойкотировать "Евровидение" в следующем году, как делали это во все годы его существования.
Loc:
"Ни для кого не секрет, что наша система здравоохранения во многом опирается на терапевтические свойства марихуаны, потому её дефицит часто является серьезной проблемой как для врачей, так и для пациентов", -- прокомментировал законодательный акт депутат парламента Юрген Хахеманн.

для: Acc = 17, Loc = 3
Acc:
Губернатор Санкт-Петербурга Георгий Полтавченко считает возможным рассмотреть идею введения обязательной про

## Задание 3 (4 балла)

Эссе

Прочитайте статью *FineWeb2: One Pipeline to Scale Them All — Adapting Pre-Training Data Processing to Every Language*.
Ссылка - https://arxiv.org/pdf/2506.20920

Напишите эссе, в котором вы, используя эту статью как конкретный пример, рассуждаете о том, что значит создать «хороший» многоязычный датасет для предобучения?

Не пересказывайте статью! Выберите что-то, что вам показалось интересным (assumption, результат эксперимента, метрику и т.п.), и постройте вокруг этого аргумент. Вы можете соглашаться с авторами, спорить с ними или добавлять что-то от себя.

Можете опираться на следующие вопросы, но можете и придумать что-то свое:

- Действительно ли возможен "one pipeline to scale them all", или некоторые аспекты отбора и очистки данных по своей природе должны быть специфичны для конкретного языка?
- Что именно означает "качество данных" в FineWeb2? Согласны ли в с этим?
- Если бы вам нужно было построить корпус для предобучения на языке, который не исследовался в статье, насколько вы бы доверяли предложенному пайплайну и где ожидали бы проблемы?
- Какое решение в дизайне FineWeb2 кажется вам наиболее убедительным — или, наоборот, наименее убедительным — и почему?
- FineWeb2 охватывает более тысячи языков, однако для многих низкоресурсных языков доступно очень мало данных, причём эти данные могут быть весьма однородными. Честно ли в этом случае заявлять об их покрытии? Как провести границу?
- В статье сначала удаляются дубликаты, а затем часть повторяющихся данных выборочно возвращается с помощью *rehydration*. То есть дубли это не всегда "плохие данные". Как тогда понять, что дедублицировать, а что нет?
- Авторы тщательно выбирают *early-signal* задачи для оценки, чтобы сравнивать разные варианты датасета. Достаточно ли качества модели на downstream-задачах для оценки качества корпуса?

Вы можете предлагать гипотетические примеры, альтернативные решения или эксперименты, с помощью которых можно было бы проверить ваш аргумент.

Цель задания — не покритиковать статью. Цель — построить связный аргумент, который нельзя было бы написать, не прочитав статью внимательно и не обдумав её содержание.

